<a href="https://colab.research.google.com/github/garam827/LLM_Study/blob/main/GEMINI_API_AI_%ED%88%AC%EC%9E%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 펑션 콜링

In [14]:
from datetime import datetime
import pytz

def get_current_time(timezone: str = 'Asia/Seoul'):
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    now_timezone = f'{now} {timezone}'
    print(now_timezone)
    print(now)
    return now_timezone


tools = [
    {
        'type' : 'function',
        'function' : {
            'name' : 'get_current_time',
            'description' : '해당 타임존의 날짜와 시간을 반환합니다.',
            'parameters' : {
                'type' : 'object',
                'properties' : {
                    'timezone' : {
                        'type' : 'string',
                        'description' : '현재 날짜와 시간을 반환할 타임존을 입력하세요.(예: Asia/Seoul)'
                    }
                },
                'required' : ['timezone'],
            }
        }
    }
]

In [15]:
get_current_time()

2026-03-25 23:02:24 Asia/Seoul
2026-03-25 23:02:24


'2026-03-25 23:02:24 Asia/Seoul'

In [16]:
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')

In [17]:
from openai import OpenAI

# Google AI Studio에서 발급받은 API 키 설정
# OpenAI SDK 형식
###### gemma 3 는 openai sdk 에서 function calling 지원 X

client = OpenAI(
    api_key = api_key,
    base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
)

response = client.chat.completions.create(
    model="gemma-3-27b-it",  # Gemma 3 27B 모델명 지정
    messages=[
        {"role": "user", "content": "너는 유능하고 친절한 AI 비서야."}, # Gemma 3 27B --> "system" role 지원 X
        {"role": "user", "content": "Gemma 3 27B의 특징을 짧게 요약해줘."}
    ],
    temperature = 0.7,
    max_tokens = 500
)

print(response.choices[0].message.content)

안녕하세요! 유능하고 친절한 AI 비서 Gemma입니다. Gemma 3 27B의 특징을 짧게 요약해 드릴게요.

Gemma 3 27B는 Google DeepMind에서 개발한 **오픈 웨이트 대형 언어 모델(LLM)**입니다. 주요 특징은 다음과 같아요:

*   **강력한 성능:** 이전 모델 대비 추론 및 코딩 능력이 향상되었습니다.
*   **다양한 작업 수행:** 텍스트 생성, 번역, 질문 답변, 코드 작성 등 다양한 작업을 수행할 수 있습니다.
*   **오픈 웨이트:** 누구나 자유롭게 사용, 수정, 배포할 수 있습니다.
*   **긴 컨텍스트 창:** 최대 8,192 토큰의 긴 컨텍스트를 처리하여 더 복잡한 작업이 가능합니다.
*   **다국어 지원:** 한국어를 포함한 다양한 언어를 지원합니다.

Gemma 3 27B는 연구 및 개발 목적으로 활용하기에 적합하며, 다양한 분야에서 혁신적인 응용 가능성을 가지고 있습니다.

더 궁금한 점이 있으시면 언제든지 질문해주세요!


In [18]:
# OpenAI SDK 형식

from openai import OpenAI
import json

client = OpenAI(
    api_key     = api_key,
    base_url    = "https://generativelanguage.googleapis.com/v1beta/openai/"
)

def get_ai_response(messages, tools = None):
    response = client.chat.completions.create(
        # model       = "gemini-3-flash-preview",
        model       = "gemini-2.5-flash-lite",
        # model       = "gemini-3.1-flash-lite",
        # model       = "gemma-3-27b-it",
        # model       = 'gemini-1.5-flash',
        # model       = 'gemini-1.5-pro-latest',
        messages    = messages,
        tools       = tools,
    )

    return response

In [20]:
# 초기 시스템 메시지 설정
messages = [
    {"role": "system",  "content": "너는 사용자를 도와주는 상담사야."},
    # {"role": "user",  "content": "너는 사용자를 도와주는 상담사야."},
]

while True:
    user_input  = input("사용자: ")

    if user_input == "exit":
        break

    messages.append({"role": "user",  "content": user_input})

    ai_response = get_ai_response(messages, tools = tools)
    ai_message = ai_response.choices[0].message
    print(ai_message)

    tool_calls = ai_message.tool_calls
    if tool_calls:
        tool_name = tool_calls[0].function.name
        tool_call_id = tool_calls[0].id

        arguments = json.loads(ai_message.tool_calls[0].function.arguments)

        if tool_name == "get_current_time":
            messages.append({
                "role": "function",
                "tool_call_id": tool_call_id,
                "name" : tool_name,
                "content": get_current_time(timezone = arguments['timezone'])
            })

        ai_response = get_ai_response(messages, tools = tools)
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)

    if ai_message.content:
        print("AI\t: " + ai_message.content)
    else:
        print("AI\t: (AI가 직접적인 텍스트 응답을 제공하지 않았습니다.)")

사용자: 안녕? 지금 몇시야?
ChatCompletionMessage(content='어느 지역의 시간을 알려드릴까요? (예: Asia/Seoul)', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
AI	: 어느 지역의 시간을 알려드릴까요? (예: Asia/Seoul)
사용자: 한국 서울!
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-13366244540961631621', function=Function(arguments='{"timezone":"Asia/Seoul"}', name='get_current_time'), type='function')])
2026-03-25 23:06:55 Asia/Seoul
2026-03-25 23:06:55
AI	: 현재 서울 시간은 2026년 3월 25일 23:06:55입니다.
사용자: 미국 LA
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-13553355527002839473', function=Function(arguments='{"timezone":"America/Los_Angeles"}', name='get_current_time'), type='function')])
2026-03-25 07:07:01 America/Lo

## 여러 도시 시간 한번에 대답

In [21]:
# 초기 시스템 메시지 설정
messages = [
    {"role": "system",  "content": "너는 사용자를 도와주는 상담사야."},
]

while True:
    user_input  = input("사용자: ")

    if user_input == "exit":
        break

    messages.append({"role": "user",  "content": user_input})

    ai_response = get_ai_response(messages, tools = tools)
    ai_message = ai_response.choices[0].message
    print(ai_message)

    tool_calls = ai_message.tool_calls
    if tool_calls:
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            tool_call_id = tool_call.id

            arguments = json.loads(tool_call.function.arguments)

            if tool_name == "get_current_time":
                messages.append({
                    "role": "function",
                    "tool_call_id": tool_call_id,
                    "name" : tool_name,
                    "content": get_current_time(timezone = arguments['timezone'])
                })
        messages.append({"role" : "system", "content" : "이제 주어진 결과를 바탕으로 답변할 차례다."})

        ai_response = get_ai_response(messages, tools = tools)
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)

    print("AI\t: " + ai_message.content)

사용자: 중국, 영국, 프랑스 시간 알려줘.
ChatCompletionMessage(content='어느 도시의 시간을 알려드릴까요?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
AI	: 어느 도시의 시간을 알려드릴까요?
사용자: 상해, 런던, 파리
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-6085252543491962552', function=Function(arguments='{"timezone":"Asia/Shanghai"}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='function-call-6085252543491961347', function=Function(arguments='{"timezone":"Europe/London"}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='function-call-6085252543491964238', function=Function(arguments='{"timezone":"Europe/Paris"}', name='get_current_time'), type='function')])
2026-03-25 22:07:24 Asia/Shanghai
2026-03-25 22:07:24
2026-03-25 14:07:24 Europe/London
2026-03-25 14:07:24
